TO-DO:

- Identify if the IQA models produced the same results or similar to that reported in their papers
- Determine which logistic5() equation to use for mapping the model scale to the dataset scales
- Identify a reliable way to detect duplicate images & images with water marks -> 2 papers in To Be Read/ 

* Duplicate_Image_Detection_in_Large_Scale_Databases - https://www.researchgate.net/publication/265233100_Duplicate_Image_Detection_in_Large_Scale_Databases/link/54b680510cf24eb34f6d263a/download?_tp=eyJjb250ZXh0Ijp7ImZpcnN0UGFnZSI6InB1YmxpY2F0aW9uIiwicGFnZSI6InB1YmxpY2F0aW9uIn19

* Finding_Groups_of_Duplicate_Images_In_Very_Large_Dataset - https://www.researchgate.net/publication/266350352_Finding_Groups_of_Duplicate_Images_In_Very_Large_Dataset

* Watermark Repo - https://github.com/frlim/data2040_final/blob/master/Project3/Network_Code.ipynb

In [1]:
import os
import re
import sys
import glob
import time
import torch
import requests
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List
from IPython.display import display

# Extra Functions

In [2]:
def create_symlink(link_path, target_path):
    '''     
    This function will create a symlink if it does not already exist.
    If the symlink already exists, it will print a message indicating that.

    Input:
    - link_path: The path where the symlink will be created.
    - target_path: The path that the symlink will point to.
    '''
    try:
        os.symlink(target_path, link_path, target_is_directory=True)
        print(f"Symlink created: {link_path} → {target_path}")
    except FileExistsError:
        print(f"Symlink already exists at: {link_path}")
    except OSError as e:
        print(f"Failed to create symlink: {e}")

In [3]:
# # Example usage
# link_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink"
# target_path = r'D:\ThesisFiles'

# create_symlink(link_path, target_path)

In [4]:
def load_row_group_from_dataset(dataset_dir, global_row_group_idx):
    """
    Loads a specific row group from a multi-file parquet dataset.

    Args:
        dataset_dir (str): Directory containing part-*.parquet files.
        global_row_group_idx (int): The global row group index across all files.

    Returns:
        pyarrow.Table: The requested row group loaded as a PyArrow Table.
    """
    # List and sort all part files
    part_files = sorted(glob.glob(os.path.join(dataset_dir, "part-*.parquet")))
    if not part_files:
        raise FileNotFoundError(f"No part-*.parquet files found in {dataset_dir}")

    # Count row groups in each file
    row_groups_counts = []
    for f in part_files:
        parquet_file = pq.ParquetFile(f)
        row_groups_counts.append(parquet_file.num_row_groups)

    # Find which file the global row group index falls into
    cumulative = 0
    for idx, count in enumerate(row_groups_counts):
        if global_row_group_idx < cumulative + count:
            # The row group is in this file
            row_group_in_file = global_row_group_idx - cumulative
            parquet_file = pq.ParquetFile(part_files[idx])
            table = parquet_file.read_row_group(row_group_in_file)
            print(f"Loaded row group {row_group_in_file} from {os.path.basename(part_files[idx])}")
            return table
        cumulative += count

    raise IndexError(f"Row group index {global_row_group_idx} out of range (max {cumulative-1})")


In [5]:
def split_parquet(input_path, output_dir, rows_per_file=2_000_000, row_group_size=200_000):
    '''
    Splits a large Parquet file into smaller files with a specified number of rows per file.

    Input:
    - input_path: Path to the input Parquet file.
    - output_dir: Directory where the split files will be saved.
    - rows_per_file: Number of rows each split file should contain.
    - row_group_size: Size of each row group in the output Parquet files.

    Output:
    - Creates multiple Parquet files in the output directory, each containing up to `rows_per_file` rows.
    '''
    os.makedirs(output_dir, exist_ok=True)
    
    # Load full table
    print(f"Reading {input_path}...")
    table = pq.read_table(input_path)
    total_rows = table.num_rows
    print(f"Total rows: {total_rows}")

    # Split and write
    for i in tqdm(range(0, total_rows, rows_per_file), desc="✂️ Splitting"):
        chunk = table.slice(i, rows_per_file)
        output_path = os.path.join(output_dir, f"part-{i // rows_per_file:05d}.parquet")
        pq.write_table(chunk, output_path, row_group_size=row_group_size)
        print(f"Wrote {output_path} with {chunk.num_rows} rows")

    print("Done!")

In [6]:
# split_parquet(
#     input_path=r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_2205000.parquet",
#     output_dir="split_output",
#     rows_per_file=2_000_000,
#     row_group_size=200_000
# )

# Download Images from .parquet 

- (w/ Retry Failed Download Logic + Continue from were left off)

In [ ]:
import pandas as pd
from pathlib import Path
import asyncio
import aiohttp
from aiohttp import ClientTimeout
from PIL import Image
from io import BytesIO
from tqdm import tqdm

async def fetch_and_save(session, image_id, url, download_dir):
    """Fetch a single image and save it to disk. Returns None if success, or (image_id, url, error) if failure."""
    image_path = download_dir / f"{image_id}.jpg"
    if image_path.exists():
        return None  # skip already downloaded

    try:
        async with session.get(url) as response:
            response.raise_for_status()
            content = await response.read()
            image = Image.open(BytesIO(content)).convert("RGB")
            image.save(image_path)
        return None
    except Exception as e:
        return (image_id, url, str(e))


async def download_images(parquet_path, download_root="./image_cache", image_name_column=None,
                          batch_size=500, concurrent_requests=128, retry_failed_downloads=False):
    """
    Download images from a Parquet file in batches, organized by Parquet filename.
    Resumes strictly from the highest existing image. Only retries missing images if retry_failed_downloads=True.
    """
    df = pd.read_parquet(parquet_path)
    print(f"Total rows loaded from {parquet_path}: {len(df)}")

    # Determine the column to use for image filenames
    if image_name_column is None:
        image_name_column = 'row_index'
        df = df.reset_index().rename(columns={'index': 'row_index'})

    # Create download directory based on parquet file name
    parquet_name = Path(parquet_path).stem
    download_dir = Path(download_root) / parquet_name
    download_dir.mkdir(parents=True, exist_ok=True)
    print(f"Images will be saved to: {download_dir}")

    # Identify existing images
    existing_images = {int(p.stem) for p in download_dir.glob("*.jpg") if p.stem.isdigit()}

    if existing_images:
        max_existing = max(existing_images)
        print(f"{len(existing_images)} images already exist. Resuming from {max_existing + 1}")
    else:
        max_existing = -1
        print("No existing images found. Starting from the beginning.")

    # Filter dataframe based on continuation logic
    if retry_failed_downloads:
        # Download all images not yet present
        df_to_download = df[~df[image_name_column].isin(existing_images)].reset_index(drop=True)
    else:
        # Strict continuation from last downloaded image
        df_to_download = df[df[image_name_column] > max_existing].reset_index(drop=True)

    print(f"{len(df_to_download)} images to download in this session.")

    failures = []
    timeout = ClientTimeout(total=15)
    connector = aiohttp.TCPConnector(limit=concurrent_requests)

    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        for batch_start in range(0, len(df_to_download), batch_size):
            batch_end = min(batch_start + batch_size, len(df_to_download))
            batch_rows = [(row[image_name_column], row["url"]) 
                          for _, row in df_to_download.iloc[batch_start:batch_end].iterrows()]
            if not batch_rows:
                continue

            tasks = [fetch_and_save(session, img_id, url, download_dir) for img_id, url in batch_rows]

            for f in tqdm(asyncio.as_completed(tasks), total=len(tasks),
                          desc=f"Downloading batch {batch_start}-{batch_end}"):
                result = await f
                if result is not None:
                    failures.append(result)

    print(f"Finished downloading. Total failures: {len(failures)}")
    return failures

In [ ]:
parquet_file = r"dataset/relaion2B-en-research-safe/0000.parquet"

failures = await download_images(
    parquet_file,
    download_root="./clip_embeddings_resumable_symlink/downloaded_images",
    image_name_column = None,
    batch_size=500,
    concurrent_requests=500,
    retry_failed_downloads = False
)

# Create CLIP Embeddings

In [7]:
def fetch_image(index_row_tuple):
    '''
    Function which retrieves an image from a URL and returns it along with its index and caption.

    Input:
    - index_row_tuple: A tuple containing the index and a dictionary with keys "url" and "caption".

    Output:    
    - A tuple containing the index, image URL, caption, and the image object (or None if the image could not be fetched).
    '''
    index, row = index_row_tuple
    image_url = row["url"]
    caption = row["caption"]

    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")

        return (index, image_url, caption, image)
    except:
        return (index, image_url, caption, None)

In [9]:
def derive_embeddings(batch, processor, model, parquet_file):
    '''
    Function to derive embeddings from a batch of images using a pre-trained model.
    Uses binary batch splitting to isolate bad images instead of falling back to full serial processing.

    Input:
    - batch: A list of tuples, each containing an index, URL, caption, and image object.
    - processor: The processor to prepare the images for the model.
    - model: The pre-trained model to derive embeddings from the images.
    - parquet_file: The name of the Parquet file from which the batch is derived.

    Output:
    - A list of dictionaries containing embedding results or None for failed cases.
    '''

    embedding_results = []
    try:
        indices, urls, captions, images = zip(*batch)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # tqdm.write(f"Processing batch of {len(batch)} images on device: {device}")

        def process_subbatch(sub_indices, sub_urls, sub_captions, sub_images):
            try:
                inputs = processor(images=list(sub_images), return_tensors="pt", padding=True).to(device)
                with torch.no_grad():
                    image_features = model.get_image_features(**inputs)
                image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                for i in range(len(sub_images)):
                    embedding_results.append({
                        'original_image_index': sub_indices[i],
                        'url': sub_urls[i],
                        'caption': sub_captions[i],
                        'embeddings_result': image_embeddings[i].cpu().numpy(),
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })
            except Exception as e:
                if len(sub_images) == 1:
                    tqdm.write(f"❌ Skipping image at index {sub_indices[0]} due to error: {e}")
                    embedding_results.append({
                        'original_image_index': sub_indices[0],
                        'url': sub_urls[0],
                        'caption': sub_captions[0],
                        'embeddings_result': None,
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })
                else:
                    mid = len(sub_images) // 2
                    process_subbatch(sub_indices[:mid], sub_urls[:mid], sub_captions[:mid], sub_images[:mid])
                    process_subbatch(sub_indices[mid:], sub_urls[mid:], sub_captions[mid:], sub_images[mid:])

        # Start processing the full batch with fault tolerance
        process_subbatch(indices, urls, captions, images)
        return embedding_results
    except Exception as e:
        print(f"Error processing batch: {e}")
        return embedding_results


- Create updated version which uses pre-downloaded images if available

In [10]:
def process_parquet_images(
    directory,
    parquet_file,
    output_base_dir,
    model_name,
    batch_size=200,
    number_of_workers=200,
    image_limit=3_000_000,
    save_interval=5000,
    dataset_split_size=2_000_000,
    keep_n_recent_saves=2,
    row_grp_size=300000,
    filter_keyword="",
):
    # Prepare output directory
    output_subdir_name = (
        f"{filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}"
        if filter_keyword else
        f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
    )
    dataset_dir = os.path.join(output_base_dir, output_subdir_name, parquet_file.replace(".parquet", "_embeddings"))
    os.makedirs(dataset_dir, exist_ok=True)

    # Load source parquet file
    parquet_path = os.path.join(directory, parquet_file)
    print(f"📅 Loading DataFrame from {parquet_path}...")
    try:
        df = pd.read_parquet(parquet_path)
        print(f"📊 Loaded {len(df)} rows.")
    except FileNotFoundError:
        raise FileNotFoundError(f"Input file not found at {parquet_path}")

    if filter_keyword:
        df = df[df["caption"].str.contains(filter_keyword, case=False, na=False)].copy()
    df = df.head(image_limit)

    # Initialize model
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # Resume tracking
    processed_indices = set()
    part_files = sorted(glob.glob(os.path.join(dataset_dir, "part-*.parquet")))

    finalized_parts = []
    for f in part_files:
        if "-checkpoint-" in f:
            continue
        try:
            table = pq.read_table(f, columns=["original_image_index"])
            processed_indices.update(table["original_image_index"].to_numpy())
            part_num = int(os.path.basename(f).split("-")[1].split(".")[0])
            finalized_parts.append(part_num)
        except:
            pass

    max_finalized_index = max(finalized_parts) + 1 if finalized_parts else 0

    # Resume from latest checkpoint if exists
    checkpoint_files = sorted(
        glob.glob(os.path.join(dataset_dir, "part-*-checkpoint-*.parquet")),
        key=os.path.getmtime,
        reverse=True
    )

    checkpoint_df = pd.DataFrame()
    checkpoint_part_index = None
    if checkpoint_files:
        latest_checkpoint = checkpoint_files[0]
        base_name = os.path.basename(latest_checkpoint)
        parts = base_name.split("-")
        try:
            checkpoint_part_index = int(parts[1])
        except:
            checkpoint_part_index = None
        try:
            table = pq.read_table(latest_checkpoint)
            checkpoint_df = table.to_pandas()
            print(f"🧭 Resuming from checkpoint part-{checkpoint_part_index:05d} with {len(checkpoint_df)} rows")
        except:
            print("⚠️ Failed to load checkpoint, starting fresh.")

    # Attempt to resume from incomplete part if no checkpoint
    if checkpoint_part_index is None and finalized_parts:
        last_part_num = max(finalized_parts)
        last_part_path = os.path.join(dataset_dir, f"part-{last_part_num:05d}.parquet")
        try:
            table = pq.read_table(last_part_path)
            last_part_df = table.to_pandas()
            if len(last_part_df) < dataset_split_size:
                checkpoint_df = last_part_df
                checkpoint_part_index = last_part_num
                split_index = last_part_num
                processed_indices.update(last_part_df["original_image_index"].to_numpy())
                print(f"🔄 Resuming incomplete part-{split_index:05d} with {len(checkpoint_df)} rows")
            else:
                split_index = last_part_num + 1
        except:
            split_index = max_finalized_index
    else:
        split_index = checkpoint_part_index if checkpoint_part_index is not None else max_finalized_index

    # Filter unprocessed rows
    df = df[~df.index.isin(processed_indices)]
    if not checkpoint_df.empty:
        checkpoint_indices = set(checkpoint_df["original_image_index"])
        df = df[~df.index.isin(checkpoint_indices)]

    print(f"📌 Remaining to process after excluding checkpoint data: {len(df)}")

    buffer = []
    total_processed = 0

    for batch_start in tqdm(range(0, len(df), batch_size), desc="Processing Batches"):
        batch_end = min(batch_start + batch_size, len(df))
        batch_rows = list(df.iloc[batch_start:batch_end].iterrows())
        
        # Downloading the batch images in parallel
        batch_download_st_time = time.time()
        with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
            fetched = list(executor.map(fetch_image, batch_rows))
        batch_download_elapsed_time = time.time() - batch_download_st_time

        valid = [i for i in fetched if i[3] is not None]
        failed = [i for i in fetched if i[3] is None]

        batch_embedding_st_time = time.time()
        results = derive_embeddings(valid, processor, model, parquet_file)
        batch_embedding_elapsed_time = time.time() - batch_embedding_st_time

        for item in failed:
            results.append({
                'original_image_index': item[0],
                'url': item[1],
                'caption': item[2],
                'embeddings_result': None,
                'similarity': None,
                'parquet_file_name': parquet_file
            })
        print(f"Batch {batch_start // batch_size + 1} processed in: Download {batch_download_elapsed_time:.2f}s | Embedding {batch_embedding_elapsed_time:.2f}s | Total {batch_download_elapsed_time + batch_embedding_elapsed_time:.2f}s - Success Links: {len(valid)} & Failed Links: {len(failed)}")

        buffer.extend(results)
        total_processed += len(results)

        # Save checkpoint
        if total_processed % save_interval < batch_size or batch_end == len(df):
            checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame(buffer)], ignore_index=True)
            checkpoint_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)
            checkpoint_df.sort_values(by='original_image_index', inplace=True)
            checkpoint_df.reset_index(drop=True, inplace=True)
            checkpoint_path = os.path.join(
                dataset_dir,
                f"part-{split_index:05d}-checkpoint-{len(checkpoint_df)}.parquet"
            )
            pq.write_table(pa.Table.from_pandas(checkpoint_df), checkpoint_path, row_group_size=row_grp_size)
            tqdm.write(f"💾 Checkpoint saved: {checkpoint_path}")
            buffer = []

            # Clean up old checkpoints
            all_checkpoints = sorted(
                glob.glob(os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")),
                key=os.path.getmtime,
                reverse=True
            )
            for old_cp in all_checkpoints[keep_n_recent_saves:]:
                try:
                    os.remove(old_cp)
                    tqdm.write(f"🗑️ Deleted old checkpoint: {old_cp}")
                except Exception as e:
                    tqdm.write(f"⚠️ Could not delete checkpoint {old_cp}: {e}")

        # Finalize part
        if len(checkpoint_df) >= dataset_split_size:
            final_path = os.path.join(dataset_dir, f"part-{split_index:05d}.parquet")
            pq.write_table(pa.Table.from_pandas(checkpoint_df), final_path, row_group_size=row_grp_size)
            tqdm.write(f"📦 Finalized: {final_path}")

            # Clean up checkpoints for this part
            checkpoint_glob = os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")
            for cp_file in glob.glob(checkpoint_glob):
                try:
                    os.remove(cp_file)
                    tqdm.write(f"🧹 Removed checkpoint: {cp_file}")
                except Exception as e:
                    tqdm.write(f"⚠️ Could not remove checkpoint {cp_file}: {e}")

            checkpoint_df = pd.DataFrame()
            split_index += 1

    # Final flush
    if not checkpoint_df.empty:
        final_path = os.path.join(dataset_dir, f"part-{split_index:05d}.parquet")
        pq.write_table(pa.Table.from_pandas(checkpoint_df), final_path, row_group_size=row_grp_size)
        tqdm.write(f"✅ Completed: {final_path}")

        checkpoint_glob = os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")
        for cp_file in glob.glob(checkpoint_glob):
            try:
                os.remove(cp_file)
                tqdm.write(f"🧹 Removed checkpoint: {cp_file}")
            except Exception as e:
                tqdm.write(f"⚠️ Could not remove checkpoint {cp_file}: {e}")


In [ ]:
# --- Configuration ---
filter_keyword = ""
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0001.parquet"
output_base_dir = "clip_embeddings_resumable_symlink"
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_index = 2
model_name = clip_model_names[model_index]
batch_size = 400 #200
number_of_workers = 100
save_interval = 5000
row_grp_size = 200_000
keep_n_recent_saves = 2
image_limit = 16_388_200

process_parquet_images(
    directory=directory,
    parquet_file=parquet_file,
    output_base_dir=output_base_dir,
    model_name=model_name,
    batch_size=batch_size,
    number_of_workers=number_of_workers,
    image_limit=image_limit,
    save_interval=save_interval,
    row_grp_size=row_grp_size,
    keep_n_recent_saves=keep_n_recent_saves,
    filter_keyword=filter_keyword,
    dataset_split_size=2_000_000,
)

# Search Images Using CLIP Embeddings

- Implement FAISS to reduce runtime

In [ ]:
def find_similar_images(
    dataset_dir: str,
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,
    similarity_threshold: Optional[float] = None,
    alpha: float = 0.5
) -> List[dict]:
    ''' 
    Functions used to find images similar to a text or image (or both) based on the CLIP embeddings.

    Inputs:
    - dataset_dir: Directory containing the CLIP embeddings dataset in Parquet format.
    - model_name: Name of the pre-trained CLIP model to use.
    - text_prompt: Optional text prompt to find similar images. (None means only the image is used to search).
    - image_path: Optional path to an image to find similar images. (None means only the text prompt is used to search).
    - top_n: Optional number of top similar images to return. (None means return all).
    - similarity_threshold: Optional threshold for cosine similarity to filter results. (None means no filtering).
    - alpha: Weight for text similarity in the combined similarity score. (0.5 - equal weight between text and image, 1 - text prompt only is used, 0 - image only is used).

    Outputs:
    - List of dictionaries containing URLs, captions, original image indices, and cosine similarities of the most similar images.
    '''

    assert text_prompt or image_path, "You must provide either a text prompt or an image path."

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text_embedding = None
    image_embedding = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy for similarity calculation
    text_embedding_np = text_embedding.cpu().squeeze().numpy() if text_embedding is not None else None
    image_embedding_np = image_embedding.cpu().squeeze().numpy() if image_embedding is not None else None

    all_matches = []

    # Loads only matching files from dataset_dir like part-00000.parquet to avoid loading checkpoints
    pattern = re.compile(r"^part-\d{5}\.parquet$")
    valid_files = [
        os.path.join(dataset_dir, f)
        for f in os.listdir(dataset_dir)
        if pattern.match(f)
    ]
    dataset = ds.dataset(valid_files, format="parquet")

    # Iterate through each fragment (.parquet file) composing the dataset
    for fragment in dataset.get_fragments():
        pq_file = pq.ParquetFile(os.path.join(dataset_dir, fragment.path))

        for row_group_index in range(pq_file.num_row_groups):
            table = pq_file.read_row_group(row_group_index)
            df_chunk = table.to_pandas()

            df_chunk = df_chunk.dropna(subset=['embeddings_result'])
            if df_chunk.empty:
                continue

            df_chunk['embeddings_result'] = df_chunk['embeddings_result'].apply(
                lambda x: np.array(x) if isinstance(x, list) else x
            )
            image_embeddings = np.vstack(df_chunk['embeddings_result'].values)

            # Compute similarities separately
            sim_text = np.dot(image_embeddings, text_embedding_np.T) if text_embedding_np is not None else 0
            sim_image = np.dot(image_embeddings, image_embedding_np.T) if image_embedding_np is not None else 0

            # Combine similarity with weights
            combined_sim = None
            if text_embedding_np is not None and image_embedding_np is not None:
                combined_sim = alpha * sim_text + (1 - alpha) * sim_image
            elif text_embedding_np is not None:
                combined_sim = sim_text
            else:
                combined_sim = sim_image

            df_chunk['combined_similarity'] = combined_sim

            if similarity_threshold is not None:
                df_chunk = df_chunk[df_chunk['combined_similarity'] >= similarity_threshold]

            for _, row in df_chunk.iterrows():
                all_matches.append({
                    'url': row.get('url'),
                    'caption': row.get('caption'),
                    'original_image_index': row.get('original_image_index'),
                    'cosine_similarity': row['combined_similarity']
                })

    all_matches.sort(key=lambda x: x['cosine_similarity'], reverse=True)
    top_matches = all_matches[:top_n] if top_n is not None else all_matches
    # top_matches.reverse()

    return top_matches #all_matches[:top_n] if top_n is not None else all_matches

In [ ]:
dataset_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "female construction worker"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 
top_n = None#10
similarity_threshold = None#0#0.3

image_info_list = find_similar_images(
    dataset_dir=dataset_dir,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path = search_image_path,
    top_n=top_n,
    similarity_threshold=similarity_threshold,
    alpha=0#1  # 1 = text only, 0 = image only, 0.5 = equal weighting
)

# TO-DO: Improve the execution time of the find_similar_images function.

# View or Download Returned Images via Search

In [12]:
def retrieve_and_show_images(
    images_info,
    save_dir=None, 
    display_images=True
):
    '''
    Function to retrieve images from URLs and display/save them.
    
    Inputs:
    - images_info: List of dictionaries containing 'url', 'caption', and 'cosine_similarity', derived from find_similar_images.
    - save_dir: Optional directory to save the images. If None, images are not saved.
    - display_images: Boolean to control whether to display images in the notebook.

    Outputs:
    - None, but prints status messages and displays images if requested.
    '''

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for idx, info in enumerate(images_info, start=1):
        url = info['url']
        caption = info.get('caption', 'No caption')
        similarity = info.get('cosine_similarity', None)
        original_index = info.get('original_image_index', None)

        try:
            response = requests.get(url)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content)).convert("RGB")

            if save_dir:
                ext = url.split('.')[-1].split('?')[0]  # crude file extension guess
                filename = f"{idx:03d}_originalIndex{original_index}_sim{similarity:.3f}.{ext}"
                filepath = os.path.join(save_dir, filename)
                img.save(filepath)
                print(f"Saved: {filepath}")

            if display_images:
                print(f"Image {idx}: {caption} (Similarity: {similarity:.4f})")
                display(img)

        except Exception as e:
            print(f"Failed to load image {idx} from {url}: {e}")

In [ ]:
save_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\retrieved_images\female_woman_image_of_construction_worker'

retrieve_and_show_images(image_info_list, save_dir=save_dir, display_images=False) #0

# Identify Duplicate Images

#### Working Find Duplicates using CLIP embeddings using IndexIVFFlat

In [ ]:
import pandas as pd
import numpy as np
import faiss
from tqdm import tqdm
import pyarrow.dataset as ds
import ast
from pathlib import Path
import random
import gc

def parse_embedding(e):
    if e is None:
        return None
    try:
        if isinstance(e, str):
            e = ast.literal_eval(e)
        return np.array(e, dtype=np.float32)
    except Exception:
        return None

def get_all_parquet_files(volume_paths):
    all_files = []
    for volume_path in tqdm(volume_paths, desc="Collecting files from volumes"):
        path = Path(volume_path)
        files_in_volume = list(path.glob('*.parquet'))
        all_files.extend(files_in_volume)
    print(f"\nFound {len(all_files)} total parquet files to process.")
    return all_files

# Works but very slow on 4Mil subset
def find_embedding_candidates_with_dataset_truly_incremental(
    parquet_files, 
    top_k=10, 
    clip_threshold=0.7, 
    batch_size=1000, 
    nlist=4096,   # IVF: number of clusters
    nprobe=16,    # IVF: number of clusters to search
    train_samples=200000  # how many vectors to sample for training
):
    print("Initializing FAISS GPU resources...")
    gpu_res = faiss.StandardGpuResources()
    cpu_index = None
    gpu_index = None
    dim = None
    
    index_to_original_id_map = []
    
    print("Creating pyarrow dataset...")
    try:
        dataset = ds.dataset(parquet_files, format="parquet")
    except Exception as e:
        print(f"Error creating dataset: {e}")
        return None
    
    total_rows = dataset.count_rows()
    print(f"Total rows in dataset: {total_rows}")

    # ----------------------------------------------------
    # Step 1: Collect a training sample for IVF
    # ----------------------------------------------------
    print(f"Sampling ~{train_samples} vectors for IVF training...")
    sample_embeddings = []
    for batch in dataset.to_batches(batch_size=batch_size):
        df = batch.to_pandas()
        df['clip_embedding'] = df['embeddings_result'].apply(parse_embedding)
        valid = df[df['clip_embedding'].notnull()]
        if len(valid) > 0:
            emb = np.stack(valid['clip_embedding'].values).astype('float32')
            sample_embeddings.append(emb)
            if sum(len(x) for x in sample_embeddings) >= train_samples:
                break
    sample_embeddings = np.vstack(sample_embeddings)[:train_samples]
    faiss.normalize_L2(sample_embeddings)
    
    dim = sample_embeddings.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    cpu_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)

    print("Training IVF index...")
    gpu_index.train(sample_embeddings)

    # ----------------------------------------------------
    # Step 2: Build the index incrementally
    # ----------------------------------------------------
    print("Building FAISS IVF index incrementally...")
    for batch in tqdm(dataset.to_batches(batch_size=batch_size*10), desc="Adding embeddings"):
        df_batch = batch.to_pandas()
        df_batch['clip_embedding'] = df_batch['embeddings_result'].apply(parse_embedding)
        df_valid = df_batch[df_batch['clip_embedding'].notnull()].reset_index(drop=True)
        if len(df_valid) == 0:
            continue
        embeddings = np.stack(df_valid['clip_embedding'].values).astype('float32')
        if embeddings.shape[1] != dim:
            continue
        faiss.normalize_L2(embeddings)
        gpu_index.add(embeddings)
        index_to_original_id_map.extend(df_valid['original_image_index'].tolist())

        del df_batch, df_valid, embeddings
        gc.collect()

    print(f"FAISS IVF index built with {gpu_index.ntotal} total vectors.")

    # ----------------------------------------------------
    # Step 3: Search in batches
    # ----------------------------------------------------
    gpu_index.nprobe = nprobe
    all_candidates = []

    print(f"Searching top-{top_k} neighbors with CLIP threshold {clip_threshold}...")
    # search_dataset = ds.dataset(parquet_files, format="parquet")

    processed_rows = 0
    # for batch in tqdm(search_dataset.to_batches(batch_size=batch_size), desc="Searching"):
    for batch in tqdm(dataset.to_batches(batch_size=batch_size), desc="Searching"):
        df_search_batch = batch.to_pandas()
        df_search_batch['clip_embedding'] = df_search_batch['embeddings_result'].apply(parse_embedding)
        # df_search_valid = df_search_batch[df_search_batch['clip_embedding'].notnull()].reset_index(drop=True)
        df_search_batch = df_search_batch[df_search_batch['clip_embedding'].notnull()].reset_index(drop=True)

        # if len(df_search_valid) == 0:
        if len(df_search_batch) == 0:
            continue

        # search_embeddings = np.stack(df_search_valid['clip_embedding'].values).astype('float32')
        search_embeddings = np.stack(df_search_batch['clip_embedding'].values).astype('float32')
        if search_embeddings.shape[1] != dim:
            continue

        faiss.normalize_L2(search_embeddings)
        D, I = gpu_index.search(search_embeddings, top_k + 1)

        for row_idx, (neighbors, sims) in enumerate(zip(I, D)):
            # src_index = df_search_valid.at[row_idx, 'original_image_index']
            src_index = df_search_batch.at[row_idx, 'original_image_index']
            for neighbor_faiss_idx, sim_score in zip(neighbors[1:], sims[1:]):  # skip self
                if neighbor_faiss_idx < 0:
                    continue
                if sim_score >= clip_threshold:
                    tgt_index = index_to_original_id_map[neighbor_faiss_idx]
                    if src_index != tgt_index:
                        sorted_pair = tuple(sorted((src_index, tgt_index)))
                        all_candidates.append((*sorted_pair, sim_score))
        processed_rows += len(df_search_batch)

    print(f"FAISS candidate search complete. {len(all_candidates)} pairs above threshold.")
    return list(set(all_candidates))

#### Working Find Duplicates using CLIP embeddings using IndexIVFPQ instead of IndexIVFFlat

In [ ]:
import pandas as pd
import numpy as np
import faiss
from tqdm import tqdm
import pyarrow.dataset as ds
import ast
from pathlib import Path
import gc

def parse_embedding(e):
    if e is None:
        return None
    try:
        if isinstance(e, str):
            e = ast.literal_eval(e)
        return np.array(e, dtype=np.float32)
    except Exception:
        return None

def get_all_parquet_files(volume_paths):
    all_files = []
    for volume_path in tqdm(volume_paths, desc="Collecting files from volumes"):
        path = Path(volume_path)
        files_in_volume = list(path.glob('*.parquet'))
        all_files.extend(files_in_volume)
    print(f"\nFound {len(all_files)} total parquet files to process.")
    return all_files


def find_embedding_candidates_with_ivfpq(
    parquet_files, 
    top_k=10, 
    clip_threshold=0.7, 
    batch_size=1000, 
    nlist=4096,     # IVF: number of clusters
    nprobe=16,      # IVF: number of clusters to search
    train_samples=200000, 
    m=32,           # number of subquantizers for PQ
    nbits=8         # bits per subquantizer (typical: 8)
):
    print("Initializing FAISS GPU resources...")
    gpu_res = faiss.StandardGpuResources()
    cpu_index = None
    gpu_index = None
    dim = None
    
    index_to_original_id_map = []
    
    print("Creating pyarrow dataset...")
    dataset = ds.dataset(parquet_files, format="parquet")
    total_rows = dataset.count_rows()
    print(f"Total rows in dataset: {total_rows}")

    # ----------------------------------------------------
    # Step 1: Collect training samples
    # ----------------------------------------------------
    print(f"Sampling ~{train_samples} vectors for IVFPQ training...")
    sample_embeddings = []
    for batch in dataset.to_batches(batch_size=batch_size):
        df = batch.to_pandas()
        df['clip_embedding'] = df['embeddings_result'].apply(parse_embedding)
        valid = df[df['clip_embedding'].notnull()]
        if len(valid) > 0:
            emb = np.stack(valid['clip_embedding'].values).astype('float32')
            sample_embeddings.append(emb)
            if sum(len(x) for x in sample_embeddings) >= train_samples:
                break
    sample_embeddings = np.vstack(sample_embeddings)[:train_samples]
    faiss.normalize_L2(sample_embeddings)
    
    dim = sample_embeddings.shape[1]

    # Build IVFPQ index
    quantizer = faiss.IndexFlatIP(dim)
    cpu_index = faiss.IndexIVFPQ(
        quantizer, dim, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT
    )
    gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)

    print("Training IVFPQ index...")
    gpu_index.train(sample_embeddings)

    # ----------------------------------------------------
    # Step 2: Add data to the index
    # ----------------------------------------------------
    print("Building FAISS IVFPQ index incrementally...")
    for batch in tqdm(dataset.to_batches(batch_size=batch_size*10), desc="Adding embeddings"):
        df_batch = batch.to_pandas()
        df_batch['clip_embedding'] = df_batch['embeddings_result'].apply(parse_embedding)
        df_valid = df_batch[df_batch['clip_embedding'].notnull()].reset_index(drop=True)
        if len(df_valid) == 0:
            continue
        embeddings = np.stack(df_valid['clip_embedding'].values).astype('float32')
        if embeddings.shape[1] != dim:
            continue
        faiss.normalize_L2(embeddings)
        gpu_index.add(embeddings)
        index_to_original_id_map.extend(df_valid['original_image_index'].tolist())

        del df_batch, df_valid, embeddings
        gc.collect()

    print(f"FAISS IVFPQ index built with {gpu_index.ntotal} total vectors.")

    # ----------------------------------------------------
    # Step 3: Search
    # ----------------------------------------------------
    gpu_index.nprobe = nprobe
    all_candidates = []

    print(f"Searching top-{top_k} neighbors with CLIP threshold {clip_threshold}...")
    for batch in tqdm(dataset.to_batches(batch_size=batch_size), desc="Searching"):
        df_search_batch = batch.to_pandas()
        df_search_batch['clip_embedding'] = df_search_batch['embeddings_result'].apply(parse_embedding)
        df_search_batch = df_search_batch[df_search_batch['clip_embedding'].notnull()].reset_index(drop=True)

        if len(df_search_batch) == 0:
            continue

        search_embeddings = np.stack(df_search_batch['clip_embedding'].values).astype('float32')
        if search_embeddings.shape[1] != dim:
            continue

        faiss.normalize_L2(search_embeddings)
        D, I = gpu_index.search(search_embeddings, top_k + 1)

        for row_idx, (neighbors, sims) in enumerate(zip(I, D)):
            src_index = df_search_batch.at[row_idx, 'original_image_index']
            for neighbor_faiss_idx, sim_score in zip(neighbors[1:], sims[1:]):
                if neighbor_faiss_idx < 0:
                    continue
                if sim_score >= clip_threshold:
                    tgt_index = index_to_original_id_map[neighbor_faiss_idx]
                    if src_index != tgt_index:
                        sorted_pair = tuple(sorted((src_index, tgt_index)))
                        all_candidates.append((*sorted_pair, sim_score))

    print(f"FAISS candidate search complete. {len(all_candidates)} pairs above threshold.")
    return list(set(all_candidates))


In [ ]:
files_to_process = [r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings\part-00000.parquet",
                    r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings\part-00001.parquet"]

all_candidates = find_embedding_candidates_with_ivfpq(
    files_to_process, 
    top_k=10, 
    clip_threshold=0.7, 
    batch_size=1000, 
    nlist=4096,     # IVF: number of clusters
    nprobe=16,      # IVF: number of clusters to search
    train_samples=200000, 
    m=32,           # number of subquantizers for PQ
    nbits=8         # bits per subquantizer (typical: 8)
)

In [ ]:
from pympler import asizeof
def get_size_info(object):
    '''Returns the size of an object in bytes and gigabytes, and prints it formatted.'''
    size_bytes = asizeof.asizeof(object)
    size_gb = size_bytes / (1024 ** 3)
    print(f"{size_gb:.2f} GB")
    return size_bytes, size_gb

get_size_info(all_candidates)

#### Working ORB Search

In [ ]:
import os
import cv2
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import groupby
from tqdm import tqdm

# -------------------- Helper --------------------
def is_valid_image(img):
    """Check if OpenCV image is non-empty and valid."""
    return (
        img is not None and
        isinstance(img, np.ndarray) and
        img.size > 0 and
        img.shape[0] > 0 and
        img.shape[1] > 0
    )

# -------------------- Streaming DSU --------------------
class StreamingDSU:
    """Memory-efficient DSU using integer mapping for streaming updates."""
    def __init__(self, elements):
        self.elem2idx = {elem: i for i, elem in enumerate(sorted(elements))}
        self.idx2elem = {i: elem for elem, i in self.elem2idx.items()}
        self.parent = np.arange(len(elements), dtype=np.int32)

    def find(self, elem):
        i = self.elem2idx[elem]
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]  # path compression
            i = self.parent[i]
        return i

    def union(self, elem1, elem2):
        i1, i2 = self.find(elem1), self.find(elem2)
        if i1 != i2:
            self.parent[i2] = i1
            return True
        return False

    def get_groups(self):
        groups = {}
        for elem, idx in self.elem2idx.items():
            root_idx = self.find(elem)
            groups.setdefault(self.idx2elem[root_idx], []).append(elem)
        return [sorted(g) for g in groups.values() if len(g) > 1]

# -------------------- ORB Verification --------------------
def verify_with_orb_streaming(
    candidates,
    image_dir,
    orb_threshold=0.35,
    nfeatures=500,
    max_workers=8,
    checkpoint_every=1000,
    checkpoint_dir="./checkpoints"
):
    image_dir = Path(image_dir)
    image_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # Sort candidates by src_idx for cache efficiency
    candidates = sorted(candidates, key=lambda x: x[0])

    # Initialize DSU, cache, invalid_images, and last processed index
    all_indices = {idx for pair in candidates for idx in pair[:2]}
    dsu = StreamingDSU(all_indices)
    orb_cache = {}
    invalid_images = set()
    last_processed_idx = -1
    processed_count = 0

    # Create single ORB extractor (reused for all images)
    orb = cv2.ORB_create(nfeatures=nfeatures)

    # Initialize FLANN for Hamming distance
    index_params = dict(algorithm=6,  # FLANN_INDEX_LSH
                        table_number=6,  # 12 is a common default
                        key_size=12,     # 20 is common
                        multi_probe_level=1)  # 2 is common
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    # -------------------- Load latest checkpoint if exists --------------------
    ckpt_files = list(checkpoint_dir.glob("dsu_checkpoint_*_*.pkl"))
    if ckpt_files:
        latest_ckpt = max(ckpt_files, key=lambda f: f.stat().st_mtime)
        processed_count = int(latest_ckpt.stem.split("_")[2])
        with open(latest_ckpt, "rb") as f:
            data = pickle.load(f)
        dsu = data['dsu']
        invalid_images = data.get('invalid_images', set())
        last_processed_idx = data.get('last_processed_idx', -1)
        candidates = [c for c in candidates if c[0] >= last_processed_idx]
        print(f"Resuming from checkpoint: {latest_ckpt}, last processed src_idx: {last_processed_idx}")

    # -------------------- Parallel ORB Extraction --------------------
    def extract_orb(idx):
        if idx in orb_cache:
            return idx, orb_cache[idx]
        img_path = image_dir / f"{idx}.jpg"
        if not img_path.exists():
            invalid_images.add(idx)
            orb_cache[idx] = (None, None)
            return idx, (None, None)
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if not is_valid_image(img):
            invalid_images.add(idx)
            orb_cache[idx] = (None, None)
            return idx, (None, None)
        try:
            kp, des = orb.detectAndCompute(img, None)
            if kp is None or des is None or len(kp) == 0:
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
        except Exception:
            invalid_images.add(idx)
            orb_cache[idx] = (None, None)
            return idx, (None, None)
        orb_cache[idx] = (kp, des)
        return idx, (kp, des)

    # -------------------- Pair Processing --------------------
    def process_pair(pair):
        src_idx, tgt_idx, _ = pair
        _, (kp1, des1) = extract_orb(src_idx)
        _, (kp2, des2) = extract_orb(tgt_idx)
        if des1 is None or des2 is None:
            return None
        try:
            matches = flann.knnMatch(des1, des2, k=2)
            good = [m for m, n in matches if len(matches) >= 2 and m.distance < 0.75 * n.distance]
            sim = len(good) / len(matches) if matches else 0.0
            if sim >= orb_threshold:
                return src_idx, tgt_idx
        except Exception:
            return None
        return None

    # -------------------- Group candidates by src_idx --------------------
    groups = groupby(candidates, key=lambda x: x[0])
    num_unique_src_ids = len(set(x[0] for x in candidates))

    for i, (src_idx, group_iter) in enumerate(tqdm(groups, total=num_unique_src_ids, desc="Streaming ORB DSU")):
        if src_idx <= last_processed_idx:
            continue

        pairs = list(group_iter)

        # Pre-extract ORB features for unique indices in this group (parallel)
        unique_idxs = {src_idx} | {tgt for _, tgt, _ in pairs}
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(extract_orb, idx): idx for idx in unique_idxs}
            for future in as_completed(futures):
                _ = future.result()  # fills orb_cache

        # Process verification in parallel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(process_pair, pairs))
        for r in results:
            if r is not None:
                dsu.union(*r)

        last_processed_idx = src_idx
        processed_count += 1

        # -------------------- Checkpoint --------------------
        if processed_count % checkpoint_every == 0:
            ckpt_file = checkpoint_dir / f"dsu_checkpoint_{processed_count}_{last_processed_idx}.pkl"
            with open(ckpt_file, "wb") as f:
                pickle.dump({
                    'dsu': dsu,
                    'invalid_images': invalid_images,
                    'last_processed_idx': last_processed_idx
                }, f)
            all_ckpts = sorted(checkpoint_dir.glob("dsu_checkpoint_*_*.pkl"),
                               key=lambda f: int(f.stem.split('_')[2]))
            for old_ckpt in all_ckpts[:-2]:
                old_ckpt.unlink(missing_ok=True)
            print(f"Checkpoint saved: {ckpt_file}, processed src_idx: {last_processed_idx}")

    # -------------------- Final groups --------------------
    final_groups = dsu.get_groups()
    print(f"Total duplicate groups found: {len(final_groups)}")
    return final_groups, invalid_images


#### Working ORB Search w CPU + GPU options (GPU requires buildings opencv-python with CUDA)

In [ ]:
import os
import cv2
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import groupby
from tqdm import tqdm

# -------------------- Helper --------------------
def is_valid_image(img):
    """Check if OpenCV image is non-empty and valid."""
    return (
        img is not None and
        isinstance(img, np.ndarray) and
        img.size > 0 and
        img.shape[0] > 0 and
        img.shape[1] > 0
    )

# -------------------- Streaming DSU --------------------
class StreamingDSU:
    """Memory-efficient DSU using integer mapping for streaming updates."""
    def __init__(self, elements):
        self.elem2idx = {elem: i for i, elem in enumerate(sorted(elements))}
        self.idx2elem = {i: elem for elem, i in self.elem2idx.items()}
        self.parent = np.arange(len(elements), dtype=np.int32)

    def find(self, elem):
        i = self.elem2idx[elem]
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]  # path compression
            i = self.parent[i]
        return i

    def union(self, elem1, elem2):
        i1, i2 = self.find(elem1), self.find(elem2)
        if i1 != i2:
            self.parent[i2] = i1
            return True
        return False

    def get_groups(self):
        groups = {}
        for elem, idx in self.elem2idx.items():
            root_idx = self.find(elem)
            groups.setdefault(self.idx2elem[root_idx], []).append(elem)
        return [sorted(g) for g in groups.values() if len(g) > 1]

# -------------------- ORB Verification --------------------
def verify_with_orb_streaming(
    candidates,
    image_dir,
    orb_threshold=0.35,
    nfeatures=500,
    max_workers=8,
    checkpoint_every=1000,
    checkpoint_dir="./checkpoints"
):
    image_dir = Path(image_dir)
    image_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # Sort candidates by src_idx for cache efficiency
    candidates = sorted(candidates, key=lambda x: x[0])

    # Initialize DSU, cache, invalid_images, and last processed index
    all_indices = {idx for pair in candidates for idx in pair[:2]}
    dsu = StreamingDSU(all_indices)
    orb_cache = {}
    invalid_images = set()
    last_processed_idx = -1
    processed_count = 0

    # -------------------- CPU/GPU ORB & Matcher --------------------
    use_cuda = cv2.cuda.getCudaEnabledDeviceCount() > 0
    if use_cuda:
        print("⚡ Using CUDA ORB + BFMatcher")
        orb = cv2.cuda_ORB.create(nfeatures=nfeatures)
        matcher = cv2.cuda.DescriptorMatcher_createBFMatcher(cv2.NORM_HAMMING)

        def extract_orb(idx):
            if idx in orb_cache:
                return idx, orb_cache[idx]
            img_path = image_dir / f"{idx}.jpg"
            if not img_path.exists():
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if not is_valid_image(img):
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            try:
                gpu_img = cv2.cuda_GpuMat()
                gpu_img.upload(img)
                kp_gpu, des_gpu = orb.detectAndComputeAsync(gpu_img, None)
                kp = orb.convert(kp_gpu)
                des = des_gpu.download() if des_gpu is not None else None
                if kp is None or des is None or len(kp) == 0:
                    invalid_images.add(idx)
                    orb_cache[idx] = (None, None)
                    return idx, (None, None)
            except Exception:
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            orb_cache[idx] = (kp, des)
            return idx, (kp, des)

        def process_pair(pair):
            src_idx, tgt_idx, _ = pair
            _, (kp1, des1) = extract_orb(src_idx)
            _, (kp2, des2) = extract_orb(tgt_idx)
            if des1 is None or des2 is None:
                return None
            try:
                d1 = cv2.cuda_GpuMat(); d2 = cv2.cuda_GpuMat()
                d1.upload(des1); d2.upload(des2)
                matches = matcher.knnMatch(d1, d2, k=2)
                good = [m for m, n in matches if m.distance < 0.75 * n.distance]
                sim = len(good) / len(matches) if matches else 0.0
                if sim >= orb_threshold:
                    return src_idx, tgt_idx
            except Exception:
                return None
            return None

    else:
        print("💻 Using CPU ORB + FLANN")
        orb = cv2.ORB_create(nfeatures=nfeatures)
        index_params = dict(algorithm=6, table_number=6, key_size=12, multi_probe_level=1)
        search_params = dict(checks=50)
        flann = cv2.FlannBasedMatcher(index_params, search_params)

        def extract_orb(idx):
            if idx in orb_cache:
                return idx, orb_cache[idx]
            img_path = image_dir / f"{idx}.jpg"
            if not img_path.exists():
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if not is_valid_image(img):
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            try:
                kp, des = orb.detectAndCompute(img, None)
                if kp is None or des is None or len(kp) == 0:
                    invalid_images.add(idx)
                    orb_cache[idx] = (None, None)
                    return idx, (None, None)
            except Exception:
                invalid_images.add(idx)
                orb_cache[idx] = (None, None)
                return idx, (None, None)
            orb_cache[idx] = (kp, des)
            return idx, (kp, des)

        def process_pair(pair):
            src_idx, tgt_idx, _ = pair
            _, (kp1, des1) = extract_orb(src_idx)
            _, (kp2, des2) = extract_orb(tgt_idx)
            if des1 is None or des2 is None:
                return None
            try:
                matches = flann.knnMatch(des1, des2, k=2)
                good = [m for m, n in matches if m.distance < 0.75 * n.distance]
                sim = len(good) / len(matches) if matches else 0.0
                if sim >= orb_threshold:
                    return src_idx, tgt_idx
            except Exception:
                return None
            return None

    # -------------------- Load latest checkpoint if exists --------------------
    ckpt_files = list(checkpoint_dir.glob("dsu_checkpoint_*_*.pkl"))
    if ckpt_files:
        latest_ckpt = max(ckpt_files, key=lambda f: f.stat().st_mtime)
        processed_count = int(latest_ckpt.stem.split("_")[2])
        with open(latest_ckpt, "rb") as f:
            data = pickle.load(f)
        dsu = data['dsu']
        invalid_images = data.get('invalid_images', set())
        last_processed_idx = data.get('last_processed_idx', -1)
        candidates = [c for c in candidates if c[0] >= last_processed_idx]
        print(f"Resuming from checkpoint: {latest_ckpt}, last processed src_idx: {last_processed_idx}")

    # -------------------- Group candidates by src_idx --------------------
    groups = groupby(candidates, key=lambda x: x[0])
    num_unique_src_ids = len(set(x[0] for x in candidates))

    for i, (src_idx, group_iter) in enumerate(tqdm(groups, total=num_unique_src_ids, desc="Streaming ORB DSU")):
        if src_idx <= last_processed_idx:
            continue

        pairs = list(group_iter)

        # Pre-extract ORB features for unique indices in this group (parallel)
        unique_idxs = {src_idx} | {tgt for _, tgt, _ in pairs}
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(extract_orb, idx): idx for idx in unique_idxs}
            for future in as_completed(futures):
                _ = future.result()  # fills orb_cache

        # Process verification in parallel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(process_pair, pairs))
        for r in results:
            if r is not None:
                dsu.union(*r)

        last_processed_idx = src_idx
        processed_count += 1

        # -------------------- Checkpoint --------------------
        if processed_count % checkpoint_every == 0:
            ckpt_file = checkpoint_dir / f"dsu_checkpoint_{processed_count}_{last_processed_idx}.pkl"
            with open(ckpt_file, "wb") as f:
                pickle.dump({
                    'dsu': dsu,
                    'invalid_images': invalid_images,
                    'last_processed_idx': last_processed_idx
                }, f)
            all_ckpts = sorted(checkpoint_dir.glob("dsu_checkpoint_*_*.pkl"),
                               key=lambda f: int(f.stem.split('_')[2]))
            for old_ckpt in all_ckpts[:-2]:
                old_ckpt.unlink(missing_ok=True)
            print(f"Checkpoint saved: {ckpt_file}, processed src_idx: {last_processed_idx}")

    # -------------------- Final groups --------------------
    final_groups = dsu.get_groups()
    print(f"Total duplicate groups found: {len(final_groups)}")
    return final_groups, invalid_images


In [ ]:
image_dir = r"clip_embeddings_resumable_symlink/downloaded_images/0000"

verify_with_orb_streaming(
    candidates=all_candidates,
    image_dir=image_dir,
    orb_threshold=0.35,
    nfeatures=500,
    max_workers=32,
    checkpoint_every=1000,
    checkpoint_dir="./0000_dsu_checkpoints_1"
)

#### Display Duplicate Images

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
import numpy as np

def show_duplicate_groups(grouped_duplicates, image_dir, max_groups=10):
    """
    Displays groups of duplicate images.

    grouped_duplicates: list of lists, where each inner list contains indices of images in a group.
    image_dir: Path to folder with images named [original_image_index].jpg
    max_groups: maximum number of groups to display
    """
    image_dir = Path(image_dir)

    for i, group in enumerate(grouped_duplicates[:max_groups]):

        if len(group) < 2:
            continue

        print(f"Displaying Group {i+1} with {len(group)} images...")
        group = group[:2]  # limit to first 20 images per group for display
        
        # Dynamically determine the grid size for the subplots
        num_images = len(group)
        cols = int(np.ceil(np.sqrt(num_images)))
        rows = int(np.ceil(num_images / cols))
        
        # Create a new figure for each group
        plt.figure(figsize=(cols * 4, rows * 4))
        plt.suptitle(f"Duplicate Group {i+1}", fontsize=16)

        for j, img_idx in enumerate(group):
            img_path = image_dir / f"{img_idx}.jpg"
            img = cv2.imread(str(img_path))

            if img is None:
                print(f"Skipping image {img_idx} – file not found.")
                continue

            # Convert BGR to RGB for matplotlib
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Add a subplot for each image in the group
            ax = plt.subplot(rows, cols, j + 1)
            ax.imshow(img)
            ax.set_title(f"Image {img_idx}")
            ax.axis('off')

        # Adjust layout to prevent titles from overlapping and display the figure
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

In [ ]:
import pickle

checkpoint_file = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\0000_dsu_checkpoints_1\dsu_checkpoint_11000_21511.pkl"

with open(checkpoint_file, "rb") as f:
    dsu = pickle.load(f)['dsu']

image_dir = r"clip_embeddings_resumable_symlink/downloaded_images/0000"
show_duplicate_groups(dsu.get_groups(), image_dir, max_groups=30)